# Forward Model Hyperparameter Tuning

Runs systematic hyperparameter search per target: for each property (UTS, Yield, Conductivity, etc.) finds the best model type (XGBoost, Random Forest, Gradient Boosting) and its best hyperparameters. Saves per-target results to `hyperparams_config.json` under `wrought.by_target` and `cast.by_target`.

In [1]:
import pandas as pd
import numpy as np
import re
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
import os

warnings.filterwarnings('ignore')

# Data paths - use local or Colab paths
WROUGHT_PATH = 'wrought_alloys_final.csv'
CAST_PATH = 'cleaned_cast_dataset.csv'

try:
    from utils import load_hyperparams, save_hyperparams, save_per_target_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, save_hyperparams, save_per_target_hyperparams

In [2]:
# Load Wrought Data
INPUT_COLS = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']

def load_wrought(path):
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        try:
            with open(path, 'rb') as f:
                head = f.read(2)
            if head == b'PK':
                df = pd.read_excel(path)  # File is xlsx with .csv extension
            else:
                try:
                    df = pd.read_csv(path, encoding='utf-8')
                except UnicodeDecodeError:
                    df = pd.read_csv(path, encoding='latin-1')
        except Exception:
            df = pd.read_excel(path)
    exclude = INPUT_COLS + ['Series', 'Parent Alloy', 'Alloy Name', 'AlloyNumber', 'Temper']
    targets = [c for c in df.columns if c not in exclude]
    for col in INPUT_COLS + targets:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df[INPUT_COLS] = df[INPUT_COLS].fillna(0.0)
    return df, targets

def load_cast(path):
    try:
        df = pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin-1')
    def extract(text):
        if pd.isna(text): return {}
        text = str(text).replace('AI', 'Al')
        m = re.search(r'\((.*?)\)', text)
        if not m: return {}
        f = m.group(1)
        comp = {}
        for el in INPUT_COLS:
            if el == 'Al': continue
            p = re.compile(rf"{el}(\d*\.?\d*)")
            h = p.search(f)
            if h:
                v = h.group(1)
                comp[el] = float(v) if (v and v != '.') else 0.5
        t = sum(comp.values())
        comp['Al'] = 100 - t if 0 < t < 100 else 0.0
        return comp
    ext = df['Alloy Name'].apply(extract)
    comp_df = pd.DataFrame(list(ext)).fillna(0.0)
    for c in INPUT_COLS:
        if c not in comp_df.columns: comp_df[c] = 0.0
    df = pd.concat([df, comp_df], axis=1)
    df = df[df['Al'] > 1.0].reset_index(drop=True)
    exclude = INPUT_COLS + ['Series', 'Parent Alloy', 'Alloy Name', 'AlloyNumber', 'Temper', 'Standard']
    targets = [c for c in df.columns if c not in exclude]
    def clean_val(val):
        if pd.isna(val): return np.nan
        s = str(val).strip()
        if '-' in s:
            try:
                parts = [re.sub(r'[^0-9.]', '', p) for p in s.split('-')]
                nums = [float(p) for p in parts if p]
                return sum(nums)/len(nums) if nums else np.nan
            except: pass
        m = re.search(r"[-+]?[0-9]*\.[0-9]+|[0-9]+", s)
        return float(m.group()) if m else np.nan
    for col in targets:
        df[col] = df[col].apply(clean_val)
    return df, targets

print('Loading datasets...')
df_wrought, targets_wrought = load_wrought(WROUGHT_PATH) if os.path.exists(WROUGHT_PATH) else (None, [])
df_cast, targets_cast = load_cast(CAST_PATH) if os.path.exists(CAST_PATH) else (None, [])
if df_wrought is not None: print(f'Wrought: {df_wrought.shape}, targets: {len(targets_wrought)}')
else: print('Wrought dataset not found')
if df_cast is not None: print(f'Cast: {df_cast.shape}, targets: {len(targets_cast)}')
else: print('Cast dataset not found')

Loading datasets...


Wrought: (868, 31), targets: 14
Cast: (41, 44), targets: 30


In [3]:
# Parameter grids for RandomizedSearchCV
XGB_PARAMS = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 3],
}
RF_PARAMS = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}
GB_PARAMS = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
}

def tune_and_select_best_per_target(df, targets, input_cols, min_samples=30):
    """
    For each target, tune XGBoost, RF, GB; pick the single best (model type + params) by R2.
    Returns by_target dict: { target_name: { 'model': 'XGBoost'|..., 'params': {...} } }
    """
    by_target = {}
    for target in targets:
        df_t = df.dropna(subset=[target])
        if len(df_t) < min_samples:
            continue
        X = df_t[input_cols]
        y = df_t[target]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        print(f">> TARGET: {target}")
        best_r2 = -float('inf')
        best_name = None
        best_params = None
        best_mae = None
        for name, model, params in [
            ('XGBoost', xgb.XGBRegressor(objective='reg:squarederror', random_state=42), XGB_PARAMS),
            ('RandomForest', RandomForestRegressor(random_state=42), RF_PARAMS),
            ('GradientBoosting', GradientBoostingRegressor(random_state=42), GB_PARAMS),
        ]:
            search = RandomizedSearchCV(model, params, n_iter=6, cv=3, verbose=0, random_state=42)
            search.fit(X_train, y_train)
            pred = search.predict(X_test)
            r2 = r2_score(y_test, pred)
            mae = mean_absolute_error(y_test, pred)
            print(f"    {name}: R²={r2:.4f}, MAE={mae:.2f}")
            if r2 > best_r2:
                best_r2 = r2
                best_name = name
                best_mae = mae
                p = {k: v for k, v in search.best_params_.items()}
                p['random_state'] = 42
                best_params = p
        if best_name is not None:
            by_target[target] = {'model': best_name, 'params': best_params}
            print(f"  WINNER for {target}: {best_name} (R²={best_r2:.4f}, MAE={best_mae:.2f})")
    return by_target

In [4]:
# Tune WROUGHT dataset (per-target best model + params)
print('[=== TUNING WROUGHT (per target) ===]')
wrought_by_target = {}
if df_wrought is not None and len(targets_wrought) > 0:
    wrought_by_target = tune_and_select_best_per_target(
        df_wrought, targets_wrought, INPUT_COLS, min_samples=30
    )
else:
    print('  Skipped (no data)')

# Tune CAST dataset (per-target best model + params)
print('\n[=== TUNING CAST (per target) ===]')
cast_targets = [t for t in targets_cast if 'Strength' in t or 'Conductivity' in t or 'Modulus' in t][:10]
if not cast_targets:
    cast_targets = targets_cast[:10]
cast_by_target = {}
if df_cast is not None and len(cast_targets) > 0:
    cast_by_target = tune_and_select_best_per_target(
        df_cast, cast_targets, INPUT_COLS, min_samples=10
    )
else:
    print('  Skipped (no data)')

[=== TUNING WROUGHT (per target) ===]
>> TARGET: UTS (MPa)


    XGBoost: R²=0.8668, MAE=35.70


    RandomForest: R²=0.8730, MAE=35.43


    GradientBoosting: R²=0.8693, MAE=35.60
  WINNER for UTS (MPa): RandomForest (R²=0.8730, MAE=35.43)
>> TARGET: YS (MPa)


    XGBoost: R²=0.6434, MAE=54.65


...

    RandomForest: R²=0.5396, MAE=10.91


    GradientBoosting: R²=0.4333, MAE=14.95
  WINNER for Electrical Conductivity: Equal Weight (% IACS): RandomForest (R²=0.5396, MAE=10.91)
>> TARGET: Strength to Weight: Axial (points)


    XGBoost: R²=-0.7307, MAE=3.03


    RandomForest: R²=-0.6260, MAE=2.90


    GradientBoosting: R²=-0.6092, MAE=2.95
  WINNER for Strength to Weight: Axial (points): GradientBoosting (R²=-0.6092, MAE=2.95)


In [5]:
# Save per-target best to config (wrought.by_target and cast.by_target)
if wrought_by_target:
    save_per_target_hyperparams('wrought', wrought_by_target)
    print(f'Saved wrought.by_target ({len(wrought_by_target)} targets) to hyperparams_config.json')
if cast_by_target:
    save_per_target_hyperparams('cast', cast_by_target)
    print(f'Saved cast.by_target ({len(cast_by_target)} targets) to hyperparams_config.json')
if not wrought_by_target and not cast_by_target:
    print('No results to save')

Saved wrought.by_target (14 targets) to hyperparams_config.json
Saved cast.by_target (9 targets) to hyperparams_config.json
